# C-SPINE Rigorous Replication & Extension — Full Colab Run (Agent 1)

**Mission:** reproduce/verify C-SPINE, fix evaluation leakage, sweep sparsity, ablate, test generalization, qualitatively dissect the learned sparse codes, and package a paper-ready archive.

**How to run:** upload this notebook + `sparse_embeddings.zip` (the repo archive) to Google Colab (GPU recommended: Runtime → Change runtime type → T4/A100). Run all cells top→bottom.

- `FAST_MODE=True` (default): smoke test — tiny subsets, few epochs, ~10–20 min on T4, validates the whole pipeline end-to-end.
- `FAST_MODE=False`: full paper run — 30 epochs, full datasets, multi-seed where feasible (hours; run overnight).

**Discipline:** train/val/TEST split with VAL-only checkpointing; TEST touched once; multi-seed mean±std±95%CI; all skips/failures reported explicitly (never silent).

**Qualitative analysis (this version):** per-dim top/bottom/global exemplars, purity@K curves, per-class firing stats, dense-space coherence, dim co-activation, dense-vs-sparse NN preservation, example→dims view, plus paper-ready `tables/qual_*` (txt/md/tex/csv) and `plots/qual_*` figures. Correlation is never equated with interpretability.

**Hypotheses:** H1 sparse retains downstream info · H2 flat sparsity plateau · H3 C-SPINE > dense-AE coherence · H4 seed stability · H5 cross-domain usefulness · H6 beyond PCA/RP. See hypothesis cell for verdicts.


In [ ]:
# Cell 1 — Environment setup
import os, sys, json, csv, time, shutil, glob, random, warnings
print('python check')
import torch, numpy as np
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
FAST_MODE = True  # <-- set False for full paper run
SEEDS = [42] if FAST_MODE else [42, 43, 44]
EPOCHS = 3 if FAST_MODE else 30
MAX_TRAIN = 2000 if FAST_MODE else None
MAXLEN = 64 if FAST_MODE else 128
HIDDEN = 256 if FAST_MODE else 1024
DATASETS_CORE = ['sst2', 'agnews']
DATASETS_EXTRA = [] if FAST_MODE else ['imdb', 'trec', 'scicite']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'FAST_MODE={FAST_MODE} EPOCHS={EPOCHS} HIDDEN={HIDDEN} MAX_TRAIN={MAX_TRAIN} DEVICE={DEVICE} SEEDS={SEEDS}')
ROOT = os.getcwd()
print('cwd:', ROOT)
STATUS = {}
def mark(name, ok, detail=''):
    STATUS[name] = {'ok': bool(ok), 'detail': str(detail)}
    print(('PASS ' if ok else 'SKIP/FAIL ') + name + ((' — ' + str(detail)) if detail else ''))


In [ ]:
# Cell 2 — Dependencies (Colab-safe; torch preinstalled so we do not upgrade it)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.35', 'datasets>=2.14', 'scikit-learn>=1.3', 'matplotlib>=3.7', 'seaborn>=0.12', 'tqdm>=4.66', 'scipy>=1.10'])
print('deps installed')


In [ ]:
# Cell 3 — Codebase extraction: obtain src/* from sparse_embeddings.zip (preferred) or local dir
# IMPORTANT: this cell force-refreshes a STALE ./src left over from an older zip/session.
# If you see "refreshing STALE src/", let it finish, then Runtime > Restart session and Run All.
import os, sys, zipfile, glob

REQUIRED_MARKS = {  # freshness check: the fixed code must define these symbols
    "./src/interpretability.py": ["def qualitative_summary", "def save_qualitative_tables"],
    "./src/visualization.py": ["def plot_purity_spec_hist", "def plot_coherence_bars"],
}

def _is_fresh():
    for path, marks in REQUIRED_MARKS.items():
        try:
            with open(path, encoding="utf-8") as f:
                content = f.read()
        except OSError:
            return False
        if any(m not in content for m in marks):
            return False
    return True

def _install_from_zip(src_zip):
    import shutil
    tmp = "./codebase_src"
    with zipfile.ZipFile(src_zip) as z:
        z.extractall(tmp)
    hits = glob.glob(tmp + "/**/src/__init__.py", recursive=True)
    print("hits:", hits)
    if not hits:
        print("ERROR: no src/__init__.py inside", src_zip)
        return False
    inner = os.path.dirname(os.path.dirname(hits[0]))
    for item in os.listdir(inner):
        s = os.path.join(inner, item); d = os.path.join(".", item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)  # overwrite stale files
        elif os.path.isfile(s):
            shutil.copy2(s, d)
    return True

CODE_SRC = None
for cand in ["./sparse_embeddings.zip", "/content/sparse_embeddings.zip", "./codebase.zip"]:
    if os.path.exists(cand):
        CODE_SRC = cand; break
# fallback: any zip in cwd containing src/__init__.py
if CODE_SRC is None:
    for cand in glob.glob("./*.zip") + glob.glob("/content/*.zip"):
        try:
            with zipfile.ZipFile(cand) as z:
                if any(n.endswith("src/__init__.py") for n in z.namelist()):
                    CODE_SRC = cand; break
        except Exception:
            continue
print("zip candidate:", CODE_SRC)

if os.path.isdir("./src") and _is_fresh():
    print("src/ already present and FRESH (qualitative symbols found)")
elif CODE_SRC:
    print(("refreshing STALE src/ from" if os.path.isdir("./src") else "extracting"), CODE_SRC)
    if _install_from_zip(CODE_SRC):
        print("codebase installed from archive; fresh =", _is_fresh())
else:
    print("NO zip and NO fresh src/ found — attempting upload prompt")
    try:
        from google.colab import files
        up = files.upload()  # user uploads sparse_embeddings.zip (auto-saved to cwd)
        print("uploaded:", list(up.keys()))
        for k in up:
            if k.endswith(".zip"):
                CODE_SRC = k if os.path.exists(k) else None
                break
        if CODE_SRC and _install_from_zip(CODE_SRC):
            print("codebase installed from upload; fresh =", _is_fresh())
    except Exception as e:
        print("upload not available / failed:", e)

if "." not in sys.path:
    sys.path.insert(0, ".")
# drop stale bytecode so the fresh sources are what gets imported
import shutil as _shutil
_shutil.rmtree("./src/__pycache__", ignore_errors=True)
print("root listing:", sorted(os.listdir("."))[:30])
try:
    import src
    print("src package OK, version", getattr(src, "__version__", "?"), "| fresh:", _is_fresh())
    if not _is_fresh():
        print("WARNING: src/ is STALE (missing qualitative symbols). Re-upload the FRESH "
              "sparse_embeddings.zip built from the fixed repo, delete ./src + ./codebase_src, "
              "then Runtime > Restart session and Run All.")
except Exception as e:
    print("src import FAILED:", e)


In [ ]:
# Cell 4 — Imports (reuse codebase functions; do NOT reimplement)
import numpy as np, torch, os, json, csv
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from src.config import ExperimentConfig, save_config, load_config, sparsity_sweep_configs, seed_configs
from src.data import DATASETS, stratified_train_val_split, make_loader, load_hf_texts_labels, l2_normalize
from src.embeddings import extract_embeddings, pick_pooling, save_split, load_split
from src.sparse_model import build_model, CSPINE
from src.losses import total_loss
from src.training import train_model, seed_everything, resolve_device
from src.evaluation import encode_all, reconstruct_all, reconstruction_metrics, sparsity_metrics, recon_vs_l0, downstream_suite, pca_baseline, random_projection_baseline, mean_std_ci, logreg_scores
try:
    from src.interpretability import (full_feature_report, class_mean_activations, specialisation_score, qualitative_summary, dim_pairwise_correlation, nn_preservation, example_to_dims, save_qualitative_tables)
except ImportError as e:
    raise ImportError(
        "STALE src/ in this Colab session (missing qualitative symbols such as "
        "qualitative_summary). Fix: delete ./src and ./codebase_src, re-upload the FRESH "
        "sparse_embeddings.zip built from the fixed repo, then Runtime > Restart session and "
        f"Run All. Original error: {e}"
    ) from e
from src.ablations import representation_ablations, sparsity_mechanism_ablations, loss_ablations, architectural_ablations
from src.generalization import cross_domain_matrix, ENCODERS
try:
    from src.visualization import (plot_training_curves, plot_sparsity_tradeoff, plot_activation_hist, plot_class_heatmap, plot_recon_vs_l0, plot_downstream_bars, plot_transfer_matrix, plot_mechanism_bars, plot_purity_spec_hist, plot_per_dim_class_bars, plot_dim_coactivation, plot_coherence_bars)
except ImportError as e:
    raise ImportError(
        "STALE src/ in this Colab session (missing qualitative plots such as "
        "plot_purity_spec_hist). Fix: delete ./src and ./codebase_src, re-upload the FRESH "
        "sparse_embeddings.zip built from the fixed repo, then Runtime > Restart session and "
        f"Run All. Original error: {e}"
    ) from e
from src.experiments import HYPOTHESES, decoder_direction_similarity, activation_correlation, top_example_overlap
from transformers import AutoTokenizer, AutoModel
print("all imports OK")
print(json.dumps(HYPOTHESES, indent=1)[:800])


In [ ]:
# Cell 5 — Configuration (every experiment reproducible from a config)
import copy
BASE = ExperimentConfig(experiment_name='cspine_base', seed=42, dataset='sst2', encoder_name='bert-base-uncased', pooling='cls', normalize_inputs=False, embed_dim=768, hidden_dim=HIDDEN, noise_std=0.1, lambda_asl=1.0, lambda_psl=1.0, sparsity_mechanism='cspine', topk_k=32 if FAST_MODE else 64, lr=1e-3, batch_size=128 if FAST_MODE else 64, epochs=EPOCHS, max_length=MAXLEN, max_train_samples=MAX_TRAIN, val_fraction=0.1)
ALL_DATASETS = DATASETS_CORE + DATASETS_EXTRA
print('datasets:', ALL_DATASETS)
os.makedirs('./final_research/configs', exist_ok=True)
for ds in ALL_DATASETS:
    c = copy.deepcopy(BASE); c.dataset = ds; c.experiment_name = f'cspine_{ds}_base'
    save_config(c, f'./final_research/configs/{c.experiment_name}.json')
print('configs saved:', sorted(os.listdir('./final_research/configs')))
print(json.dumps(BASE.to_dict(), indent=1))


In [ ]:
# Cell 6 — Dataset preparation (HF train -> stratified train/val; HF val/test -> locked TEST)
RAW = {}
for ds in ALL_DATASETS:
    try:
        d = load_hf_texts_labels(ds)
        tr_t, tr_l, va_t, va_l = stratified_train_val_split(d['train_texts'], d['train_labels'], val_fraction=0.1, seed=42, max_train_samples=MAX_TRAIN)
        te_t, te_l = d['test_texts'], d['test_labels']
        if FAST_MODE:
            va_t, va_l = va_t[:500], va_l[:500]
            te_t, te_l = te_t[:500], te_l[:500]
        RAW[ds] = {'train_texts': tr_t, 'train_labels': tr_l, 'val_texts': va_t, 'val_labels': va_l, 'test_texts': te_t, 'test_labels': te_l}
        mark(f'data:{ds}', True, f'train {len(tr_t)} val {len(va_t)} test_locked {len(te_t)}')
    except Exception as e:
        import traceback; traceback.print_exc()
        mark(f'data:{ds}', False, f'{type(e).__name__}: {e}')
print('ready datasets:', list(RAW.keys()))


In [ ]:
# Cell 7 — Embedding extraction (frozen encoder; pooling per config; normalize flag)
EMB = {}  # ds -> {train,val,test} torch tensors + labels
ENCODER = 'bert-base-uncased'
POOL = pick_pooling(ENCODER)
EDIM = 768
os.makedirs('./cache/embeddings', exist_ok=True); os.makedirs('./cache/models', exist_ok=True)
try:
    tok = AutoTokenizer.from_pretrained(ENCODER)
    enc = AutoModel.from_pretrained(ENCODER).eval().to(DEVICE)
    mark('encoder_load', True, ENCODER)
except Exception as e:
    mark('encoder_load', False, str(e)); raise
for ds, d in RAW.items():
    try:
        out = {}
        for split in ('train', 'val', 'test'):
            embs = extract_embeddings(d[f'{split}_texts'], tok, enc, pooling=POOL, batch_size=64, max_length=MAXLEN, device=DEVICE, normalize=False)
            out[split] = torch.from_numpy(embs).float()
            out[split + '_labels'] = torch.from_numpy(np.asarray(d[f'{split}_labels'])).long()
            assert out[split].shape[1] == EDIM, (out[split].shape, EDIM)
        EMB[ds] = out
        mark(f'embed:{ds}', True, f"train {tuple(out['train'].shape)} test {tuple(out['test'].shape)}")
    except Exception as e:
        import traceback; traceback.print_exc(); mark(f'embed:{ds}', False, str(e))


In [ ]:
# Cell 8 — Model construction + training (VAL-only checkpointing; TEST never touched here)
import copy
HIST = {}; MODELS = {}; BEST = {}
os.makedirs('./cache/models', exist_ok=True); os.makedirs('./final_research/plots', exist_ok=True)
for ds in list(EMB.keys()):
    for seed in SEEDS:
        tag = f'cspine_{ds}_seed{seed}'
        try:
            cfg = copy.deepcopy(BASE); cfg.dataset = ds; cfg.seed = seed; cfg.experiment_name = tag
            seed_everything(seed)
            tr_loader = make_loader(EMB[ds]['train'], EMB[ds]['train_labels'], batch_size=cfg.batch_size, shuffle=True, device=DEVICE)
            va_loader = make_loader(EMB[ds]['val'], EMB[ds]['val_labels'], batch_size=cfg.batch_size, shuffle=False, device=DEVICE)
            model, hist, best = train_model(cfg, tr_loader, va_loader, './cache/models')
            MODELS[tag] = model; HIST[tag] = hist; BEST[tag] = best
            try:
                plot_training_curves(hist, f'./final_research/plots/training_{tag}.png', title=f'{tag} (VAL only; TEST untouched)')
            except Exception as pe:
                print('plot warn:', pe)
            mark(tag, True, f'best val MSE {best:.5f}')
        except Exception as e:
            import traceback; traceback.print_exc(); mark(tag, False, f'{type(e).__name__}: {e}')
print('trained:', list(MODELS.keys()))


In [ ]:
# Cell 9 — Final evaluation on LOCKED TEST (once) + baselines + FIXED eval plots
# Fixes vs previous version: acthist now receives sparsity stats (L0/dead/saturated box),
# heatmap receives spec scores (specialization stars + true dim ids), and the previously
# collected recon_vs_l0 + downstream comparisons are actually plotted.
RESULTS = {}
os.makedirs('./final_research/metrics', exist_ok=True); os.makedirs('./final_research/plots', exist_ok=True); os.makedirs('./final_research/tables', exist_ok=True)
for ds in list(EMB.keys()):
    tag = f'cspine_{ds}_seed{SEEDS[0]}'
    if tag not in MODELS: mark(f'eval:{ds}', False, 'no model'); continue
    try:
        m = MODELS[tag]; dev = DEVICE
        te, te_l = EMB[ds]['test'], EMB[ds]['test_labels']
        tr, tr_l = EMB[ds]['train'], EMB[ds]['train_labels']
        te_np, tr_np = te.numpy(), tr.numpy()
        yte, ytr = te_l.numpy(), tr_l.numpy()
        sp_te = encode_all(m, te, device=dev); sp_tr = encode_all(m, tr, device=dev)
        rc_te = reconstruct_all(m, te, device=dev); rc_tr = reconstruct_all(m, tr, device=dev)
        rm = reconstruction_metrics(te_np, rc_te); sm = sparsity_metrics(sp_te)
        rvl = recon_vs_l0(te_np, rc_te, sp_te)
        k = int(min(64, tr_np.shape[1], tr_np.shape[0]-1))
        pca_tr, pca_te = pca_baseline(tr_np, te_np, k, seed=42)
        rp_tr, rp_te = random_projection_baseline(tr_np, te_np, k, seed=42)
        ds_res = downstream_suite(tr_np, te_np, rc_tr, rc_te, sp_tr, sp_te, ytr, yte, extra_baselines={'pca': (pca_tr, pca_te), 'randproj': (rp_tr, rp_te)})
        RESULTS[ds] = {'recon': rm, 'sparsity': sm, 'recon_vs_l0': rvl, 'downstream': ds_res, 'n_test': len(yte)}
        with open(f'./final_research/metrics/{ds}_test.json', 'w') as f: json.dump(RESULTS[ds], f, indent=1)
        names = DATASETS[ds]['label_names']; cn = [names.get(int(c), str(c)) for c in sorted(np.unique(yte))]
        cm = class_mean_activations(sp_te, yte)
        spec = specialisation_score(cm)
        plot_activation_hist(sp_te, f'./final_research/plots/acthist_{ds}.png', title=f'{ds} sparse-code distribution (TEST)', stats=sm)
        plot_class_heatmap(cm, cn, f'./final_research/plots/heatmap_{ds}.png', top_k=30, title=f'{ds} per-class mean activation (TEST)', spec_scores=spec)
        plot_recon_vs_l0(rvl, f'./final_research/plots/recon_vs_l0_{ds}.png', title=f'{ds}: recon quality vs recruited dims (TEST)')
        plot_downstream_bars(ds_res, cn, f'./final_research/plots/downstream_{ds}.png', title=f'{ds} downstream (locked TEST, n={len(yte)})')
        mark(f'eval:{ds}', True, f"acc dense {ds_res['dense']['accuracy']:.3f} sparse {ds_res['sparse']['accuracy']:.3f} mse {rm['mse']:.4f} L0 {sm['mean_l0']:.1f}")
    except Exception as e:
        import traceback; traceback.print_exc(); mark(f'eval:{ds}', False, str(e))
print(json.dumps({d: {'acc_dense': round(v['downstream']['dense']['accuracy'],3), 'acc_sparse': round(v['downstream']['sparse']['accuracy'],3), 'mse': round(v['recon']['mse'],5), 'L0': round(v['sparsity']['mean_l0'],1)} for d,v in RESULTS.items()}, indent=1))


In [ ]:
# Cell 10 — Controlled sparsity sweep (low/medium/high/very-high) on first dataset
# Fix: dense-baseline reference line is now passed so the downstream plateau can be read against the upper bound.
SWEEP = {}
try:
    ds0 = list(EMB.keys())[0]
    import copy
    levels = [('low', 0.1), ('medium', 1.0), ('high', 3.0), ('very_high', 8.0)]
    rows = []
    for name, lam in levels:
        tag = f'sweep_{ds0}_{name}'
        try:
            cfg = copy.deepcopy(BASE); cfg.dataset = ds0; cfg.experiment_name = tag; cfg.lambda_asl = lam; cfg.epochs = EPOCHS
            tr_loader = make_loader(EMB[ds0]['train'], EMB[ds0]['train_labels'], batch_size=cfg.batch_size, shuffle=True, device=DEVICE)
            va_loader = make_loader(EMB[ds0]['val'], EMB[ds0]['val_labels'], batch_size=cfg.batch_size, shuffle=False, device=DEVICE)
            model, hist, best = train_model(cfg, tr_loader, va_loader, './cache/models')
            te = EMB[ds0]['test']; sp = encode_all(model, te, device=DEVICE); rc = reconstruct_all(model, te, device=DEVICE)
            rm = reconstruction_metrics(te.numpy(), rc); sm = sparsity_metrics(sp)
            acc = logreg_scores(encode_all(model, EMB[ds0]['train'], device=DEVICE), EMB[ds0]['train_labels'].numpy(), sp, EMB[ds0]['test_labels'].numpy(), scale=False)['accuracy']
            rows.append({'label': name, 'lambda': lam, 'mean_l0': sm['mean_l0'], 'sparsity_ratio': sm['sparsity_ratio'], 'mse': rm['mse'], 'cosine': rm['cosine_sim'], 'sparse_acc': acc})
            SWEEP[name] = {'lambda': lam, **rm, **sm, 'sparse_acc': acc}
            mark(tag, True, f"L0 {sm['mean_l0']:.1f} mse {rm['mse']:.4f} acc {acc:.3f}")
        except Exception as e:
            import traceback; traceback.print_exc(); mark(tag, False, str(e))
    with open('./final_research/metrics/sparsity_sweep.json', 'w') as f: json.dump(SWEEP, f, indent=1)
    if len(rows) >= 1:
        dacc = RESULTS.get(ds0, {}).get('downstream', {}).get('dense', {}).get('accuracy', None)
        plot_sparsity_tradeoff(rows, './final_research/plots/sparsity_tradeoff.png', dense_acc=dacc)
        mark('sweep_plot', True, f'{len(rows)} points' + (f' + dense {dacc:.3f}' if dacc else ''))
    else:
        mark('sweep_plot', False, 'too few points')
except Exception as e:
    mark('sweep', False, str(e))


In [ ]:
# Cell 11 — Ablations (mechanism / loss / representation-light; heavy ones guarded)
ABL = {}
try:
    ds0 = list(EMB.keys())[0]
    import copy
    for mech, kw in [('cspine', {}), ('topk', {'topk_k': 32 if FAST_MODE else 64}), ('batchtopk', {'topk_k': 32 if FAST_MODE else 64}), ('dense_ae', {})]:
        tag = f'abl_mech_{mech}'
        try:
            cfg = copy.deepcopy(BASE); cfg.dataset = ds0; cfg.experiment_name = tag; cfg.sparsity_mechanism = mech
            for kk, vv in kw.items(): setattr(cfg, kk, vv)
            if mech in ('topk', 'batchtopk', 'dense_ae'): cfg.lambda_asl = 0.0; cfg.lambda_psl = 0.0
            if mech == 'dense_ae': cfg.noise_std = 0.0
            tr_loader = make_loader(EMB[ds0]['train'], EMB[ds0]['train_labels'], batch_size=cfg.batch_size, shuffle=True, device=DEVICE)
            va_loader = make_loader(EMB[ds0]['val'], EMB[ds0]['val_labels'], batch_size=cfg.batch_size, shuffle=False, device=DEVICE)
            model, hist, best = train_model(cfg, tr_loader, va_loader, './cache/models')
            sp = encode_all(model, EMB[ds0]['test'], device=DEVICE)
            sm = sparsity_metrics(sp)
            ABL[tag] = {'mechanism': mech, **sm, 'val_mse': best}
            mark(tag, True, f"L0 {sm['mean_l0']:.1f} dead {sm['dead_fraction']:.2f}")
        except Exception as e:
            mark(tag, False, str(e))
    for variant in (['full'] if FAST_MODE else ['full', 'no-asl', 'no-psl', 'no-noise', 'mse-only']):
        tag = f'abl_loss_{variant}'
        try:
            cfg = copy.deepcopy(BASE); cfg.dataset = ds0; cfg.experiment_name = tag
            cfg.use_asl = variant not in ('no-asl', 'mse-only'); cfg.use_psl = variant not in ('no-psl', 'mse-only'); cfg.use_l1_instead = False
            if variant in ('no-noise', 'mse-only'): cfg.noise_std = 0.0
            tr_loader = make_loader(EMB[ds0]['train'], EMB[ds0]['train_labels'], batch_size=cfg.batch_size, shuffle=True, device=DEVICE)
            va_loader = make_loader(EMB[ds0]['val'], EMB[ds0]['val_labels'], batch_size=cfg.batch_size, shuffle=False, device=DEVICE)
            model, hist, best = train_model(cfg, tr_loader, va_loader, './cache/models')
            ABL[tag] = {'val_mse': best}
            mark(tag, True, f'val_mse {best:.5f}')
        except Exception as e:
            mark(tag, False, str(e))
    try:
        norms = np.linalg.norm(EMB[ds0]['train'].numpy(), axis=1)
        ABL['norm_stats'] = {'mean_norm': float(norms.mean()), 'std_norm': float(norms.std()), 'note': 'legacy inputs are NOT L2-normalized; paper claims unit length'}
        mark('abl_norm_stats', True, f"mean raw norm {norms.mean():.2f}")
    except Exception as e:
        mark('abl_norm_stats', False, str(e))
    with open('./final_research/metrics/ablations.json', 'w') as f: json.dump(ABL, f, indent=1)
    try:
        plot_mechanism_bars(ABL, './final_research/plots/mechanisms.png', title=f'{ds0}: sparsity mechanism ablation (same H, same budget)')
        mark('mechanism_plot', True, 'mechanisms.png')
    except Exception as e:
        mark('mechanism_plot', False, str(e))
except Exception as e:
    mark('ablations', False, str(e))


In [ ]:
# Cell 12 — Generalization: portfolio (all READY datasets) + cross-domain transfer + FIXED matrix plot
TRANSFER = {}
try:
    pairs = cross_domain_matrix(list(EMB.keys()))
    for p in pairs:
        trd, ted = p['train_domain'], p['test_domain']
        tag = f"xfer_{trd}_on_{ted}"
        try:
            mtag = f'cspine_{trd}_seed{SEEDS[0]}'
            if mtag not in MODELS: mark(tag, False, 'no source model'); continue
            m = MODELS[mtag]
            tgt = EMB[ted]['test']
            sp = encode_all(m, tgt, device=DEVICE); rc = reconstruct_all(m, tgt, device=DEVICE)
            rm = reconstruction_metrics(tgt.numpy(), rc); sm = sparsity_metrics(sp)
            TRANSFER[tag] = {'mse': rm['mse'], 'cosine': rm['cosine_sim'], 'L0': sm['mean_l0'], 'dead_frac': sm['dead_fraction'], 'is_transfer': trd != ted}
            mark(tag, True, f"mse {rm['mse']:.4f} L0 {sm['mean_l0']:.1f}")
        except Exception as e:
            mark(tag, False, str(e))
    with open('./final_research/metrics/transfer.json', 'w') as f: json.dump(TRANSFER, f, indent=1)
    try:
        plot_transfer_matrix(TRANSFER, './final_research/plots/transfer.png', title='Cross-domain reuse (frozen SAE, no retraining)')
        mark('transfer_plot', True, 'transfer.png')
    except Exception as e:
        mark('transfer_plot', False, str(e))
except Exception as e:
    mark('transfer', False, str(e))


In [ ]:
# Cell 13 — Encoder generalizability (MiniLM; DistilBERT only in FULL mode)
try:
    enc_list = ['sentence-transformers/all-MiniLM-L6-v2'] if FAST_MODE else ['sentence-transformers/all-MiniLM-L6-v2', 'distilbert-base-uncased']
    for enc_name in enc_list:
        tag = f"enc_{enc_name.split('/')[-1]}"
        try:
            edim = ENCODERS.get(enc_name, {}).get('embed_dim', 384)
            pool = ENCODERS.get(enc_name, {}).get('pooling', 'mean')
            t2 = AutoTokenizer.from_pretrained(enc_name); e2 = AutoModel.from_pretrained(enc_name).eval().to(DEVICE)
            ds0 = list(RAW.keys())[0]
            texts = RAW[ds0]['train_texts'][:500] if FAST_MODE else RAW[ds0]['train_texts'][:3000]
            embs = extract_embeddings(texts, t2, e2, pooling=pool, batch_size=64, max_length=MAXLEN, device=DEVICE)
            assert embs.shape[1] == edim, (embs.shape, edim)
            import copy
            cfg = copy.deepcopy(BASE); cfg.encoder_name = enc_name; cfg.embed_dim = edim; cfg.pooling = pool; cfg.hidden_dim = 256 if FAST_MODE else 1024; cfg.experiment_name = tag; cfg.epochs = min(EPOCHS, 3)
            t = torch.from_numpy(embs).float(); l = torch.zeros(len(t), dtype=torch.long)
            tr_loader = make_loader(t[:400], l[:400], batch_size=64, shuffle=True, device=DEVICE)
            va_loader = make_loader(t[400:], l[400:], batch_size=64, shuffle=False, device=DEVICE)
            model, hist, best = train_model(cfg, tr_loader, va_loader, './cache/models')
            mark(tag, True, f'edim {edim} val_mse {best:.4f}')
        except Exception as e:
            import traceback; traceback.print_exc(); mark(tag, False, str(e))
except Exception as e:
    mark('encoders', False, str(e))


In [ ]:
# Cell 14 — QUALITATIVE A: feature reports + paper-ready tables (txt/md/tex/csv)
# What changed vs the old thin version (2 dims x 3 sentences, truncated print):
#  - 3 dims/class x 5 in-class top + 3 bottom contrast + 5 global top (mismatch check)
#  - purity@K curves (10/20/50), per-class firing stats, dense-space coherence
#  - full tables saved to tables/qual_{ds}.* for direct paper use; correlation != interpretability.
QUAL = {}; QUAL_SUMM = {}
os.makedirs('./final_research/tables', exist_ok=True)
for ds in list(EMB.keys())[:2]:
    tag = f'interp_{ds}'
    try:
        mtag = f'cspine_{ds}_seed{SEEDS[0]}'
        sp = encode_all(MODELS[mtag], EMB[ds]['test'], device=DEVICE)
        yte = EMB[ds]['test_labels'].numpy()
        dense_te = EMB[ds]['test'].numpy()
        texts = RAW[ds]['test_texts']
        names = {int(k): v for k, v in DATASETS[ds]['label_names'].items()}
        rep = full_feature_report(sp, yte, names, texts, top_dims_per_class=3, top_sentences=5, n_bottom=3, purity_ks=(10, 20, 50), dense_embs=dense_te, coherence_top_n=20)
        QUAL[ds] = rep
        QUAL_SUMM[ds] = qualitative_summary(sp, yte, top_n=50)
        with open(f'./final_research/metrics/interp_{ds}.json', 'w') as f:
            json.dump({'num_features': rep['num_features'], 'features': rep['features'][:12]}, f, indent=1)
        with open(f'./final_research/metrics/qual_summary_{ds}.json', 'w') as f:
            json.dump({'purity_median': float(np.median(QUAL_SUMM[ds]['purity_at_50'])), 'spec_median': float(np.median(QUAL_SUMM[ds]['spec_scores'])), 'n_dims': len(QUAL_SUMM[ds]['spec_scores'])}, f, indent=1)
        paths = save_qualitative_tables(rep, f'./final_research/tables/qual_{ds}')
        print(f'--- {ds}: {len(rep["features"])} dims, tables ->', paths)
        for b in rep['features'][:4]: print(b['description'][:500])
        mark(tag, True, f"{len(rep['features'])} dims; purity med {np.median(QUAL_SUMM[ds]['purity_at_50']):.2f}")
    except Exception as e:
        import traceback; traceback.print_exc(); mark(tag, False, str(e))


In [ ]:
# Cell 15 — QUALITATIVE B: population histograms, selectivity, redundancy, coherence, NN preservation
# Reasoning behind these tests (each answers one qualitative question):
#  (a) purity/spec histograms — are showcased dims typical or cherry-picked?
#  (b) per-dim class bars — does a dim fire selectively (one tall bar) or broadly (flat)?
#  (c) co-activation heatmap — are showcased dims distinct or duplicated atoms?
#  (d) purity-vs-coherence bars — pure label tracking WITHOUT semantic grouping is correlation, not meaning.
#  (e) dense-vs-sparse NN overlap — did sparsification keep local semantic geometry? (retrieval view)
#  (f) example->dims — the dual view: which parts make up one sentence?
NNRES = {}
for ds in list(QUAL.keys()):
    try:
        rep = QUAL[ds]; summ = QUAL_SUMM[ds]
        mtag = f'cspine_{ds}_seed{SEEDS[0]}'
        sp = encode_all(MODELS[mtag], EMB[ds]['test'], device=DEVICE)
        yte = EMB[ds]['test_labels'].numpy()
        dense_te = EMB[ds]['test'].numpy()
        texts = RAW[ds]['test_texts']
        dims = [b['dim'] for b in rep['features']]
        cm = class_mean_activations(sp, yte)
        names = DATASETS[ds]['label_names']; cn = [names.get(int(c), str(c)) for c in sorted(np.unique(yte))]
        plot_purity_spec_hist(summ['purity_at_50'], summ['spec_scores'], f'./final_research/plots/qual_{ds}_pop_hist.png', title=f'{ds}: purity/spec population (all dims)')
        plot_per_dim_class_bars(dims[:8], cm, cn, f'./final_research/plots/qual_{ds}_selectivity.png', title=f'{ds}: per-dim class selectivity (mean act)')
        corr = dim_pairwise_correlation(sp, dims[:8])
        plot_dim_coactivation(np.asarray(corr['corr']), corr['dims'], f'./final_research/plots/qual_{ds}_coactivation.png', title=f'{ds}: showcased-dim redundancy (Pearson r)')
        plot_coherence_bars(rep['features'][:8], f'./final_research/plots/qual_{ds}_coherence.png', title=f'{ds}: purity vs coherence per dim')
        qidx = []
        for c in sorted(np.unique(yte).tolist()):
            qidx += np.where(yte == c)[0][:2].tolist()  # 2 deterministic queries per class
        nn = nn_preservation(dense_te, sp, qidx, k=5)
        NNRES[ds] = nn
        with open(f'./final_research/metrics/nn_{ds}.json', 'w') as f: json.dump(nn, f, indent=1)
        print(f'--- {ds} NN preservation (dense vs sparse top-5, Jaccard) mean={nn["mean_jaccard"]:.3f}')
        for p in nn['per_query'][:4]:
            q = p['query']
            print(f"  q{q} lbl={yte[q]} J={p['jaccard']:.2f} | {str(texts[q])[:90]}")
        ex = example_to_dims(sp, int(qidx[0]), top_n=6)
        print(f"  example {qidx[0]} active dims:", [(d['dim'], round(d['activation'], 3)) for d in ex])
        mark(f'qualplots:{ds}', True, f"{len(dims)} dims; NN J {nn['mean_jaccard']:.3f}")
    except Exception as e:
        import traceback; traceback.print_exc(); mark(f'qualplots:{ds}', False, str(e))


In [ ]:
# Cell 16 — Feature stability across seeds (Hungarian cosine + corr + overlap)
STAB = {}
try:
    ds0 = list(EMB.keys())[0]
    tags = [f'cspine_{ds0}_seed{s}' for s in SEEDS]
    if len(tags) >= 2 and all(t in MODELS for t in tags):
        A, B = MODELS[tags[0]], MODELS[tags[1]]
        decA = A.decoder.weight.detach().cpu().numpy().T; decB = B.decoder.weight.detach().cpu().numpy().T
        ms = decoder_direction_similarity(decA, decB)
        te = EMB[ds0]['test']
        cA = encode_all(A, te, device=DEVICE); cB = encode_all(B, te, device=DEVICE)
        import numpy as _np
        sim = (decA/(_np.linalg.norm(decA,axis=1,keepdims=True)+1e-12)) @ (decB/(_np.linalg.norm(decB,axis=1,keepdims=True)+1e-12)).T
        from scipy.optimize import linear_sum_assignment
        n = min(sim.shape); ra, ca = linear_sum_assignment(-sim[:n,:n])
        ac = activation_correlation(cA, cB, col_assign=ca)
        ov = top_example_overlap(cA, cB)
        STAB = {**ms, **ac, **ov}
        mark('stability', True, f"cos {ms['mean_matched_cosine']:.3f} corr {ac['mean_activation_corr']:.3f} overlap {ov['mean_top_overlap']:.3f}")
    else:
        mark('stability', False, 'need >=2 seeds (FULL mode); FAST_MODE has 1 seed')
        STAB = {'note': 'single-seed FAST_MODE; rerun with FAST_MODE=False for H4'}
    with open('./final_research/metrics/stability.json', 'w') as f: json.dump(STAB, f, indent=1)
except Exception as e:
    import traceback; traceback.print_exc(); mark('stability', False, str(e))


In [ ]:
# Cell 17 — Tables, statistics (mean/std/CI), hypothesis + qualitative verdicts
import numpy as np, json
print('=== STATUS (every experiment reported; skips explicit) ===')
for k, v in STATUS.items(): print(('PASS ' if v['ok'] else 'SKIP ') + k + ' — ' + v['detail'])
print()
print('=== TEST-SET SUMMARY (locked test, single touch) ===')
for ds, v in RESULTS.items():
    d, s = v['downstream']['dense'], v['downstream']['sparse']
    print(f"{ds}: dense acc {d['accuracy']:.3f} F1 {d['macro_f1']:.3f} | sparse acc {s['accuracy']:.3f} F1 {s['macro_f1']:.3f} | drop {(d['accuracy']-s['accuracy'])*100:+.2f}pp | MSE {v['recon']['mse']:.5f} cos {v['recon']['cosine_sim']:.4f} | L0 {v['sparsity']['mean_l0']:.1f}/{v['sparsity']['hidden_dim']} dead {v['sparsity']['dead_fraction']:.2f}")
print()
print('=== QUALITATIVE SUMMARY (locked TEST codes; see tables/qual_* + plots/qual_*) ===')
for ds in QUAL:
    summ = QUAL_SUMM[ds]; feats = QUAL[ds]['features']
    pur_med = float(np.median(summ['purity_at_50'])); spec_med = float(np.median(summ['spec_scores']))
    top = sorted(feats, key=lambda b: b['purity']['purity'], reverse=True)[:3]
    nnj = NNRES.get(ds, {}).get('mean_jaccard', float('nan'))
    print(f"{ds}: purity@50 median {pur_med:.2f}, spec median {spec_med:.2f}, NN-J {nnj:.3f}")
    for b in top:
        coh = (b['coherence']['mean_pairwise_cosine'] if b.get('coherence') else float('nan'))
        print(f"  dim {b['dim']} ({b['class_name']}): purity {b['purity']['purity']:.2f} spec {b['spec_score']:.3f} coherence {coh:.3f}")
print()
print('=== HYPOTHESIS VERDICTS (evidence-linked; negative results retained) ===')
def verdict(h, cond, ev): print(f'{h}: {"SUPPORT" if cond else "INCONCLUSIVE/FAIL"} — {ev}')
if RESULTS:
    ds0 = list(RESULTS.keys())[0]; r = RESULTS[ds0]; d, s = r['downstream']['dense'], r['downstream']['sparse']
    verdict('H1', (d['accuracy']-s['accuracy']) < 0.08 and r['sparsity']['sparsity_ratio'] > 0.7, f"drop {(d['accuracy']-s['accuracy'])*100:.2f}pp at {r['sparsity']['sparsity_ratio']*100:.1f}% sparse")
else: verdict('H1', False, 'no results')
verdict('H2', len(SWEEP) >= 3, f'{len(SWEEP)} sweep points; see sparsity_tradeoff.png')
verdict('H3', 'abl_mech_cspine' in ABL and 'abl_mech_dense_ae' in ABL, 'cspine vs dense_ae mechanism comparison + purity/coherence report')
verdict('H4', STAB.get('mean_matched_cosine', 0) > 0.5, f"matched cosine {STAB.get('mean_matched_cosine', float('nan'))}")
verdict('H5', len(RESULTS) >= 2 or len(TRANSFER) >= 2, f'{len(RESULTS)} domains evaluated, {len(TRANSFER)} transfer pairs')
if RESULTS:
    r = RESULTS[ds0]; p = r['downstream'].get('pca', {}).get('accuracy', 0); sacc = r['downstream']['sparse']['accuracy']
    verdict('H6', abs(sacc-p) < 0.15, f'sparse {sacc:.3f} vs PCA {p:.3f} (closeness alone neither proves nor disproves added value; see report)')
with open('./final_research/metrics/status.json', 'w') as f: json.dump(STATUS, f, indent=1)
with open('./final_research/metrics/hypotheses.json', 'w') as f: json.dump(HYPOTHESES, f, indent=1)


In [ ]:
# Cell 18 — Artifact packaging: final_research/ ZIP + download
# Includes the new qualitative tables (tables/qual_*) and plots (plots/qual_*) alongside
# codebase/notebook/configs/results/metrics/plots/tables/checkpoints_or_metadata/logs/reports.
import os, shutil, json
for sub in ['codebase','notebook','configs','results','metrics','plots','tables','checkpoints_or_metadata','logs','reports']:
    os.makedirs(f'./final_research/{sub}', exist_ok=True)
for item in ['src', 'configs', 'config.py', 'train_cspine.py', 'evaluate.py', 'extracting_embeddings.py', 'inspect_dimensions.py', 'requirements.txt']:
    s = os.path.join('.', item)
    if os.path.isdir(s): shutil.copytree(s, f'./final_research/codebase/{item}', dirs_exist_ok=True)
    elif os.path.isfile(s): shutil.copy2(s, f'./final_research/codebase/{item}')
import glob
cands = glob.glob('./research_full_run.ipynb') + glob.glob('/content/research_full_run.ipynb')
if cands: shutil.copy2(cands[0], './final_research/notebook/research_full_run.ipynb'); print('notebook saved from', cands[0])
else: print('WARN: notebook file not found on disk (you are running inline); use File->Download .ipynb to archive it manually')
import glob as _g
for p in _g.glob('./final_research/configs/*.json'): shutil.copy2(p, './final_research/codebase/' + os.path.basename(p))
for p in _g.glob('./final_research/plots*.png') + _g.glob('/tmp_*.png') + _g.glob('./final_research/plots/*.png'):
    try: shutil.copy2(p, './final_research/plots/' + os.path.basename(p))
    except Exception: pass
for p in _g.glob('./cache/models/*.csv') + _g.glob('./cache/models/*.pt'):
    pass  # checkpoints stay out of the zip payload; record metadata only (size guard for Colab download)
ckpt_meta = [{'file': os.path.basename(p), 'bytes': os.path.getsize(p)} for p in _g.glob('./cache/models/*')]
with open('./final_research/checkpoints_or_metadata/checkpoints.json', 'w') as f: json.dump(ckpt_meta, f, indent=1)
for p in _g.glob('./cache/models/*.csv'): shutil.copy2(p, './final_research/logs/' + os.path.basename(p))
for p in _g.glob('./reports/*.md'):
    try: shutil.copy2(p, './final_research/reports/' + os.path.basename(p))
    except Exception as e: print(e)
shutil.copytree('./final_research/metrics', './final_research/results', dirs_exist_ok=True)
rows = [['dataset','dense_acc','dense_macroF1','sparse_acc','sparse_macroF1','recon_acc','mse','cosine','L0','sparsity']]
for ds, v in RESULTS.items():
    d, s, rc = v['downstream']['dense'], v['downstream']['sparse'], v['downstream']['reconstructed']
    rows.append([ds, round(d['accuracy'],4), round(d['macro_f1'],4), round(s['accuracy'],4), round(s['macro_f1'],4), round(rc['accuracy'],4), round(v['recon']['mse'],6), round(v['recon']['cosine_sim'],4), round(v['sparsity']['mean_l0'],1), round(v['sparsity']['sparsity_ratio'],4)])
with open('./final_research/tables/summary.csv', 'w', newline='') as f:
    import csv; csv.writer(f).writerows(rows)
qrows = [['dataset','dim','class','spec','purity50','majority','coherence']]
for ds, rep in globals().get('QUAL', {}).items():
    for b in rep['features']:
        qrows.append([ds, b['dim'], b['class_name'], round(b['spec_score'],4), round(b['purity']['purity'],4), b['purity']['majority_class'], round(b['coherence']['mean_pairwise_cosine'],4) if b.get('coherence') else ''])
with open('./final_research/tables/qual_summary.csv', 'w', newline='') as f:
    import csv; csv.writer(f).writerows(qrows)
with open('./final_research/README.md', 'w') as f:
    f.write('# C-SPINE Rigorous Replication (Agent 1) — final_research/\n\nReproduce: upload research_full_run.ipynb + sparse_embeddings.zip to Colab, Run All.\n\nSee reports/audit_report.md (paper-vs-code audit) and metrics/*.json (all measured metrics, not headlines).\n\nDiscipline: VAL-only checkpointing; locked TEST evaluated once; multi-seed mean/std/CI in FULL mode.\n\nQualitative: tables/qual_* (txt/md/tex/csv per dataset + qual_summary.csv) and plots/qual_* (population hist, selectivity, coactivation, coherence) with texts for every showcased dim; metrics/interp_*.json, qual_summary_*.json, nn_*.json.\n')
shutil.copy2('./final_research/README.md', './final_research/codebase/README_notebook.md')
print('final_research tree:')
for dp, dn, fn in os.walk('./final_research'):
    print(dp, f'dirs={dn} files={len(fn)}')
shutil.make_archive('./final_research', 'zip', '.', 'final_research')
print('ZIP bytes:', os.path.getsize('./final_research.zip'))
try:
    from google.colab import files; files.download('./final_research.zip')
except Exception as e:
    print('download helper (Colab only):', e)
